# ANALYSE-02b : L'entropie de Shannon — la marche avant ANALYSE-03

**Série** : SymbolicAI / Lean — Digestions de résultats profonds, companion du capstone [Lean-20 — Digestions Tao](../Lean-20-Capstone-Digestions-Tao-Python.ipynb)
**Rôle** : marche nommée du parcours (règle 5 de la gradation, #17545) — le prérequis que [ANALYSE-03](ANALYSE-03-PFR-Lean.ipynb) et [ANALYSE-04](ANALYSE-04-PFR-Primitives-Python.ipynb) supposent sans jamais le poser
**Kernel** : `python3` — aucune preuve formelle ici : ce carnet mesure, ANALYSE-03 énonce et cite le lac

## Navigation

| Carnet précédent | Carnet suivant |
|---|---|
| [ANALYSE-02 — *Analysis I* de Tao](ANALYSE-02-Tao-Lean-Python.ipynb) | [ANALYSE-03 — PFR](ANALYSE-03-PFR-Lean.ipynb) |

***

## Pourquoi ce carnet

Le README de la sous-série nomme deux marches à écrire ; celle-ci est la première. La raison est mesurable en ouvrant les carnets voisins :

| Carnet | Ce qu'il fait de l'entropie | Ce qu'il ne fait pas |
|---|---|---|
| ANALYSE-03 §3 | énonce le pont `H(X) = log\|A\|`, la distance de Ruzsa, l'inégalité de Ruzsa | ne définit jamais `H`, ne calcule aucune entropie |
| ANALYSE-03 §5 | décrit `Real.binEntropy` et ses lemmes Mathlib | suppose la notion déjà connue |
| ANALYSE-04 §1 | traite Ruzsa comme une **primitive** à tester, avec calcul et test de limite | utilise `entropy_bits` comme une boîte noire |

Les trois traitent donc l'entropie comme un acquis. Ce carnet est cet acquis : on y définit `H`, on la calcule, on vérifie ses deux propriétés de travail (règle de chaîne, sous-modularité) et on s'arrête exactement là où ANALYSE-03 commence.

## Ce que ce carnet n'est pas

- **Pas une digestion de lac** : aucun `#check`, aucun lac compilé. Les énoncés formels et leurs axiomes restent dans ANALYSE-03.
- **Pas une preuve** : tout ce qui suit est vérifié par le calcul sur des lois finies, ce qui donne des certitudes sur des cas, jamais un théorème. La section 5 dit précisément ce que le calcul attrape et ce qu'il laisse au lac.
- **Pas un carnet de Ruzsa** : la distance de Ruzsa est le sujet d'ANALYSE-04. On ne la définit ici que pour montrer d'où elle sort.

## Plan

1. L'entropie, définie et mesurée — bits et nats
2. Entropie = cardinal vu au logarithme
3. Entropie jointe, conditionnelle, règle de chaîne
4. Sous-modularité et information mutuelle
5. Ce que la marche transmet à ANALYSE-03
6. Exercices
7. Conclusion


## 1. L'entropie, définie et mesurée

### 1.1 Deux unités, une seule quantité

Pour une variable aléatoire `X` à valeurs dans un ensemble fini, de loi `p(x) = P(X = x)` :

```
H(X) = − Σ_x p(x) · log p(x)          (convention 0 · log 0 = 0)
```

Le choix de la base du logarithme ne change que l'unité : en base 2 on compte en **bits**, en base `e` en **nats**. C'est la seule différence entre les deux conventions, et elle explique une apparente incohérence du dépôt : ANALYSE-03 §5.1 annonce `binEntropy` « en nats » et cite le maximum `log 2 ≈ 0,693`, alors que les calculs de ce carnet sont en bits et donnent un maximum de `1`. Les deux sont justes, l'unité diffère d'un facteur `ln 2`.

Trois propriétés se lisent sur la formule, et le calcul ci-dessous les vérifie :

- `H(X) ≥ 0`, avec égalité si et seulement si `X` est presque sûrement constante ;
- `H(X) ≤ log |support(X)|`, avec égalité si et seulement si `X` est uniforme sur son support ;
- l'entropie ne dépend que de la **loi**, jamais des étiquettes.


In [1]:
import math
from collections import Counter


def entropy_bits(dist):
    """Entropie de Shannon en bits d'une loi {symbole: probabilite}."""
    return -sum(p * math.log2(p) for p in dist.values() if p > 0) + 0.0


def entropy_nats(dist):
    """Meme quantite, en nats (base e) -- cf. Real.binEntropy de Mathlib."""
    return -sum(p * math.log(p) for p in dist.values() if p > 0) + 0.0


def joint_from_conditional(px, py_given_x):
    """Loi jointe (x, y) a partir de p(x) et de p(y | x)."""
    joint = {}
    for x, p in px.items():
        for y, q in py_given_x(x).items():
            if p * q > 0:
                joint[(x, y)] = p * q
    return joint


def marginal(joint, index):
    """Loi marginale d'une jointe {(x, y): p}, selon l'indice 0 (x) ou 1 (y)."""
    out = Counter()
    for key, p in joint.items():
        out[key[index]] += p
    return dict(out)


print("Helpers charges : entropy_bits, entropy_nats, joint_from_conditional, marginal")
print("Facteur bits -> nats : ln(2) =", round(math.log(2), 6))

Helpers charges : entropy_bits, entropy_nats, joint_from_conditional, marginal
Facteur bits -> nats : ln(2) = 0.693147


In [2]:
# Piece biaisee : H(p) pour p de 0 a 1 -- les trois proprietes de 1.1 en une table.
print(f"{'p':>5} {'H en bits':>10} {'H en nats':>10}")
for i in range(0, 11):
    p = i / 10
    dist = {0: 1 - p, 1: p}
    print(f"{p:>5.1f} {entropy_bits(dist):>10.4f} {entropy_nats(dist):>10.4f}")

fair = entropy_bits({0: 0.5, 1: 0.5})
degenerate = entropy_bits({0: 1.0})
h_nats = entropy_nats({0: 0.5, 1: 0.5})
print()
print("Piece equilibree : H =", fair, "bit  (maximum sur un support de 2)")
print("Piece certaine   : H =", degenerate, "bit  (minimum, support de taille 1)")
print("Meme piece en nats : H =", round(h_nats, 6), "nat")
print("Conversion : 1 bit =", round(math.log(2), 6), "nat ; 1 nat =",
      round(1 / math.log(2), 6), "bit")

    p  H en bits  H en nats
  0.0     0.0000     0.0000
  0.1     0.4690     0.3251
  0.2     0.7219     0.5004
  0.3     0.8813     0.6109
  0.4     0.9710     0.6730
  0.5     1.0000     0.6931
  0.6     0.9710     0.6730
  0.7     0.8813     0.6109
  0.8     0.7219     0.5004
  0.9     0.4690     0.3251
  1.0     0.0000     0.0000

Piece equilibree : H = 1.0 bit  (maximum sur un support de 2)
Piece certaine   : H = 0.0 bit  (minimum, support de taille 1)
Meme piece en nats : H = 0.693147 nat
Conversion : 1 bit = 0.693147 nat ; 1 nat = 1.442695 bit


### Lecture du résultat

La table est la fonction `H₂` d'une pièce biaisée, et elle porte les trois propriétés annoncées :

- **bornes** : `H` part de `0` en `p = 0` et `p = 1` (pièce certaine) et culmine à `1` bit en `p = 0,5` — le maximum `log 2` d'un support de deux éléments, atteint exactement à l'uniformité ;
- **symétrie** : `H(p) = H(1 − p)`, visible en lisant la table à l'envers — remplacer « pile » par « face » ne change rien à l'incertitude ;
- **unités** : les dernières lignes mesurent la conversion — `1 bit = ln 2 = 0,6931 nat`, et `1 nat = 1,4427 bit`. C'est ce qui réconcilie la colonne du carnet avec le `binEntropy ≈ 0,693` de Mathlib cité par ANALYSE-03 : les deux décrivent la même pièce équilibrée, l'un en nats, l'autre en bits, et `0,693` **est** `1` exprimé en nats. Une lecture qui comparerait directement `0,693` et `1` conclurait à une erreur, alors qu'il n'y a qu'un changement d'unité.

Le point à retenir pour la suite : `H` est maximale à l'uniforme **sur un support fixé**. C'est cette propriété qui va devenir, à la section suivante, l'égalité `H = log |A|`.


## 2. Entropie = cardinal vu au logarithme

### 2.1 Le pont

Si `X` est uniforme sur un ensemble fini `A` de cardinal `|A|`, chaque atome porte la probabilité `1/|A|`, et la définition donne immédiatement :

```
H(X) = − Σ_{x ∈ A} (1/|A|) · log |A|^{-1} = log |A|
```

C'est tout le « pli informationnel » d'ANALYSE-03 §3.1 : **l'entropie d'une loi uniforme est le logarithme du cardinal de son support**. Un énoncé combinatoire qui ne parle que de cardinaux peut donc se réécrire en entropies, et réciproquement.

Deux conséquences que le calcul va montrer :

1. `H` **ne voit pas la structure** : trois groupes d'ordre 8 — cyclique, produit de trois copies de `Z/2Z`, produit `Z/4Z × Z/2Z` — donnent la même entropie `log 8`, parce qu'ils ont le même cardinal. La différence entre ces groupes n'apparaîtra qu'en regardant des **sommes** de variables (section 5).
2. `H` d'une loi uniforme sur un ensemble qui n'est **pas** un sous-groupe vaut `log |A|` sans que `A` ait la moindre structure — `log 3` pour `{0, 1, 2} ⊂ Z/8Z`. L'entropie est aveugle à l'algèbre.


In [3]:
# Trois groupes d'ordre 8, et un ensemble sans structure -- meme H, cardinal seul decide.
groupes = {
    "Z/8Z (cyclique)": [(x,) for x in range(8)],
    "(Z/2Z)^3": [(a, b, c) for a in range(2) for b in range(2) for c in range(2)],
    "Z/4Z x Z/2Z": [(a, b) for a in range(4) for b in range(2)],
}

for nom, elements in groupes.items():
    n = len(elements)
    uniforme = {e: 1 / n for e in elements}
    print(f"{nom:<16} |G| = {n}  H = {entropy_bits(uniforme):.4f} bits  log2|G| = {math.log2(n):.4f}")

print()
A = [0, 1, 2]
uniforme_A = {a: 1 / len(A) for a in A}
print(f"A = {A} (pas un sous-groupe de Z/8Z)  |A| = {len(A)}")
print(f"  H(X) = {entropy_bits(uniforme_A):.4f} bits   log2|A| = {math.log2(len(A)):.4f}")
print()
# Etiquettes arbitraires : H ne depend que de la loi, jamais des noms.
renomme = {"alpha": 0.25, "beta": 0.5, "gamma": 0.25}
print("Renommage {0:0.25, 1:0.5, 2:0.25} -> {'alpha','beta','gamma'} :",
      round(entropy_bits({0: 0.25, 1: 0.5, 2: 0.25}), 6),
      "==", round(entropy_bits(renomme), 6))

Z/8Z (cyclique)  |G| = 8  H = 3.0000 bits  log2|G| = 3.0000
(Z/2Z)^3         |G| = 8  H = 3.0000 bits  log2|G| = 3.0000
Z/4Z x Z/2Z      |G| = 8  H = 3.0000 bits  log2|G| = 3.0000

A = [0, 1, 2] (pas un sous-groupe de Z/8Z)  |A| = 3
  H(X) = 1.5850 bits   log2|A| = 1.5850

Renommage {0:0.25, 1:0.5, 2:0.25} -> {'alpha','beta','gamma'} : 1.5 == 1.5


### Lecture du résultat

Les trois premières lignes donnent **exactement la même valeur**, `3,0000` bits, alors que les trois groupes n'ont pas la même structure : `Z/8Z` a un élément d'ordre 8, `(Z/2Z)³` n'a que des éléments d'ordre 2, et `Z/4Z × Z/2Z` est intermédiaire. C'est la mesure du point 1 : à distribution uniforme fixée, `H` ne dépend que du cardinal.

La ligne suivante montre l'autre face : `H = log₂ 3 = 1,5850` pour `A = {0, 1, 2}` — un ensemble qui n'est pas un sous-groupe de `Z/8Z`. L'entropie n'exige aucune structure pour être définie et vaut quand même `log |A|`. C'est précisément ce qui rend le pli informationnel utile : il transporte une information de **taille**, pas d'algèbre — et c'est aussi sa limite, car une inégalité comme `|A + A| ≤ K|A|` parle, elle, de l'algèbre du groupe.

La dernière ligne vérifie la troisième propriété de la section 1 : renommer les symboles ne change pas `H`. L'entropie est une fonction de la loi, pas de l'étiquetage.

**Ce qu'il faut en garder** : `H(uniforme sur A) = log |A|` n'est pas une convention mais une identité, et elle va servir deux fois — pour traduire la croissance `|A + A|` en inégalité d'entropies (section 5), et pour lire `entropy_le_log_card` du lac, qui n'est que la version supérieure de cette identité.


## 3. Entropie jointe, conditionnelle, règle de chaîne

### 3.1 Les définitions utiles

Pour deux variables `X` et `Y` :

```
H(X, Y) = − Σ p(x, y) · log p(x, y)                    entropie jointe
H(Y | X) = H(X, Y) − H(X)                             entropie conditionnelle
H(X, Y) = H(X) + H(Y | X)                             règle de chaîne
```

La règle de chaîne n'est pas un théorème profond — c'est la définition de `H(Y|X)` réécrite — mais c'est **la** brique de la méthode entropique : elle permet de décomposer l'information d'un couple en « ce que `X` apporte » plus « ce que `Y` ajoute, `X` connu ». ANALYSE-04 en tire sa primitive 2 (décomposition projection / fibres), et c'est aussi sous cette forme que la preuve de PFR empile ses conditions.

Une conséquence immédiate, utile pour lire les inégalités du lac : `H(Y|X) ≤ H(Y)`, avec égalité si et seulement si `X` et `Y` sont indépendantes. « Conditionner ne peut qu'informer » — c'est-à-dire retirer de l'incertitude.

L'exemple ci-dessous est un **canal binaire symétrique** : `X` uniforme sur `{0, 1}`, et `Y = X` avec probabilité `q`, `1 − X` sinon. Aucun tirage aléatoire n'est nécessaire — la loi jointe s'écrit à la main, donc le résultat est reproductible à l'octet.


In [4]:
# Canal binaire symetrique : Y = X avec probabilite q, 1-X sinon (q = fidelite du canal).
def canal_binaire(q):
    px = {0: 0.5, 1: 0.5}
    return joint_from_conditional(px, lambda x: {x: q, 1 - x: 1 - q})


print(f"{'q':>5} {'H(X)':>7} {'H(Y)':>7} {'H(X,Y)':>8} {'H(Y|X)':>8} {'chaine OK':>10}")
for q in (0.5, 0.7, 0.9, 1.0):
    joint = canal_binaire(q)
    hx = entropy_bits(marginal(joint, 0))
    hy = entropy_bits(marginal(joint, 1))
    hxy = entropy_bits(joint)
    hy_given_x = hxy - hx
    ok = abs(hxy - (hx + hy_given_x)) < 1e-12
    print(f"{q:>5.1f} {hx:>7.4f} {hy:>7.4f} {hxy:>8.4f} {hy_given_x:>8.4f} {str(ok):>10}")

print()
print("H2(q) en bits, la quantite attendue pour H(Y|X) :")
for q in (0.5, 0.7, 0.9, 1.0):
    print(f"  q = {q}: H2 = {entropy_bits({0: 1 - q, 1: q}):.4f}")

    q    H(X)    H(Y)   H(X,Y)   H(Y|X)  chaine OK
  0.5  1.0000  1.0000   2.0000   1.0000       True
  0.7  1.0000  1.0000   1.8813   0.8813       True
  0.9  1.0000  1.0000   1.4690   0.4690       True
  1.0  1.0000  1.0000   1.0000   0.0000       True

H2(q) en bits, la quantite attendue pour H(Y|X) :
  q = 0.5: H2 = 1.0000
  q = 0.7: H2 = 0.8813
  q = 0.9: H2 = 0.4690
  q = 1.0: H2 = 0.0000


### Lecture du résultat

La colonne « chaîne OK » vaut `True` sur les quatre lignes : `H(X, Y) = H(X) + H(Y|X)` est vérifiée au bit près, ce qui est attendu — la règle **est** la définition de la conditionnelle, écrite autrement. Ce que la table apprend, c'est la lecture des deux extrêmes :

- **`q = 0,5`** — le canal est une pièce indépendante : `H(X, Y) = 2` bits, `H(Y|X) = 1` bit. Connaître `X` n'apprend **rien** sur `Y`. C'est le cas d'égalité `H(Y|X) = H(Y)` annoncé en 3.1, et la définition de l'indépendance vue par l'entropie.
- **`q = 1`** — le canal est parfait : `H(X, Y) = 1` bit, `H(Y|X) = 0`. `Y` est entièrement déterminée par `X`, l'entropie conditionnelle s'annule.

Entre les deux, `H(Y|X)` décroît avec `q`, et la dernière série de lignes l'identifie exactement : `H(Y|X) = H₂(q)`, l'entropie binaire de la section 1. C'est un point qui vaut la peine d'être vu, parce qu'il relie les trois sections : une entropie conditionnelle **est** une entropie de la section 1, prise sur la loi du bruit du canal.

**Conséquence pour la suite** : `H(Y|X) = 0` veut dire « `X` détermine `Y` ». Les preuves entropiques utilisent cette lecture en sens inverse — montrer qu'une entropie conditionnelle est petite, c'est montrer qu'une variable est presque déterminée par une autre, donc que la structure cherchée existe presque.


## 4. Sous-modularité et information mutuelle

### 4.1 La seule inégalité dont tout le reste dépend

On définit l'**information mutuelle** par

```
I(X ; Y) = H(X) + H(Y) − H(X, Y) = H(Y) − H(Y | X)
```

et la sous-modularité de l'entropie s'écrit :

```
H(X, Y) ≤ H(X) + H(Y)                    avec égalité ⟺ X ⊥ Y
```

C'est l'inégalité la plus utilisée de la théorie, et c'est aussi celle qui porte le nom le plus trompeur : elle dit simplement qu'un couple n'apporte jamais plus d'information que ses deux membres séparés, et que le déficit `I(X;Y)` mesure exactement ce que les deux variables ont en commun.

Le calcul ci-dessous vérifie les deux bornes sur trois régimes, et surtout le **cas d'égalité** — c'est lui qui compte, parce qu'une inégalité dont on ne sait pas quand elle est stricte ne permet aucune conclusion de structure.


In [5]:
def information_mutuelle(joint):
    """I(X;Y) = H(X) + H(Y) - H(X,Y)."""
    hx = entropy_bits(marginal(joint, 0))
    hy = entropy_bits(marginal(joint, 1))
    hxy = entropy_bits(joint)
    return hx + hy - hxy


cas = {
    "independantes (q=0.5)": canal_binaire(0.5),
    "correlees (q=0.9)": canal_binaire(0.9),
    "deterministes (q=1.0)": canal_binaire(1.0),
}

print(f"{'regime':<24} {'H(X,Y)':>8} {'H(X)+H(Y)':>10} {'borne':>7} {'I(X;Y)':>8}")
for nom, joint in cas.items():
    hx = entropy_bits(marginal(joint, 0))
    hy = entropy_bits(marginal(joint, 1))
    hxy = entropy_bits(joint)
    print(f"{nom:<24} {hxy:>8.4f} {hx + hy:>10.4f} {str(hxy <= hx + hy + 1e-12):>7} "
          f"{information_mutuelle(joint):>8.4f}")

print()


def marginal_2(joint3, keep):
    """Loi marginale de la jointe a trois variables, en gardant les indices `keep`."""
    out = Counter()
    for cle, p in joint3.items():
        out[tuple(cle[i] for i in keep)] += p
    return dict(out)


# Sous-modularite a trois variables : H(X,Z) + H(Y,Z) >= H(X,Y,Z) + H(Z)
print("Sous-modularite, verifiee sur le canal q=0.9 avec Z = X xor Y :")
joint3 = {(x, y, x ^ y): p for (x, y), p in canal_binaire(0.9).items()}
h_xyz = entropy_bits(joint3)
h_xz = entropy_bits(marginal_2(joint3, (0, 2)))
h_yz = entropy_bits(marginal_2(joint3, (1, 2)))
h_z = entropy_bits(marginal_2(joint3, (2,)))
gauche, droite = h_xz + h_yz, h_xyz + h_z
print(f"  H(X,Z) + H(Y,Z) = {h_xz:.4f} + {h_yz:.4f} = {gauche:.4f}")
print(f"  H(X,Y,Z) + H(Z) = {h_xyz:.4f} + {h_z:.4f} = {droite:.4f}")
print(f"  inegalite tenue : {gauche >= droite - 1e-12}   exces = {gauche - droite:.4f} bit")

regime                     H(X,Y)  H(X)+H(Y)   borne   I(X;Y)
independantes (q=0.5)      2.0000     2.0000    True   0.0000
correlees (q=0.9)          1.4690     2.0000    True   0.5310
deterministes (q=1.0)      1.0000     2.0000    True   1.0000

Sous-modularite, verifiee sur le canal q=0.9 avec Z = X xor Y :
  H(X,Z) + H(Y,Z) = 1.4690 + 1.4690 = 2.9380
  H(X,Y,Z) + H(Z) = 1.4690 + 0.4690 = 1.9380
  inegalite tenue : True   exces = 1.0000 bit


### Lecture du résultat

La colonne « borne » vaut `True` sur les trois régimes : `H(X, Y) ≤ H(X) + H(Y)` tient, y compris à la limite. Ce que la table isole, c'est **où** elle est stricte :

- **indépendantes** : `H(X, Y) = 2 = H(X) + H(Y)`, donc `I = 0`. L'égalité est le critère d'indépendance — pas seulement une borne atteinte par hasard ;
- **corrélées** : `H(X, Y) = 1,469 < 2`, avec `I = 0,531` bit. Le déficit mesure la redondance entre les deux variables ;
- **déterministes** : `H(X, Y) = 1`, `I = 1` bit, c'est-à-dire exactement `H(X)`. Une variable qui détermine l'autre partage **toute** son information : l'information mutuelle ne peut pas dépasser `min(H(X), H(Y))`.

Le régime déterministe est celui qui porte la lecture de structure : quand une preuve entropique montre que `I` est proche de son maximum, elle montre qu'une variable détermine presque l'autre — et c'est ainsi que la méthode de PFR transforme « `A` est presque un sous-groupe » en « une variable est presque déterminée par une autre ».

La dernière vérification porte sur la sous-modularité à trois variables, dans sa forme `H(X, Z) + H(Y, Z) ≥ H(X, Y, Z) + H(Z)` — celle que le lac utilise sous le nom de `entropy_submodular`. L'excès mesuré n'est pas une petite quantité : il vaut **exactement 1 bit**, et ce n'est pas un hasard. Comme `Z = X XOR Y`, connaître `Z` ne laisse pas `X` et `Y` indépendantes : `Z` connu, `Y` est déterminé par `X`, donc `I(X ; Y | Z) = H(Y | Z) = 1` bit. L'excès de la sous-modularité **est** cette information mutuelle conditionnelle — c'est la lecture qui rend l'inégalité utilisable dans les preuves, où l'on cherche précisément à borner ce que deux variables savent l'une de l'autre une fois une troisième connue.

**Une réserve honnête** : ce calcul confirme la forme de l'inégalité sur un cas, il ne prouve ni la sous-modularité en général, ni le fait que la forme soit la bonne pour toutes les variables — c'est exactement ce que le lac établit, et le calcul ne peut pas le remplacer.


## 5. Ce que la marche transmet à ANALYSE-03

### 5.1 Traduire une croissance de cardinal en inégalité d'entropies

On a maintenant les trois pièces : `H(uniforme sur A) = log |A|` (section 2), la règle de chaîne (section 3) et la sous-modularité (section 4). Elles suffisent à écrire le pont d'ANALYSE-03 §3.1 : si `X` et `X'` sont indépendantes et uniformes sur `A`, alors `X + X'` suit une loi portée par `A + A`, donc

```
H(X + X') ≤ log |A + A|
```

et de même `H(X − X') ≤ log |A − A|`. Une hypothèse de croissance `|A + A| ≤ K |A|` se réécrit donc en `H(X + X') ≤ log K + H(X)` — l'énoncé combinatoire devient un énoncé sur des entropies, où les outils de la section 4 s'appliquent.

Le calcul ci-dessous mesure les deux membres sur un `A` concret, et cherche délibérément un cas où l'inégalité est **stricte** : c'est celui qui enseigne le plus, parce qu'il montre que le pont est une borne et non une identité.


In [6]:
def distribution_somme(A, n):
    """Loi de X + X' pour X, X' independantes uniformes sur A (entiers modulo n)."""
    counts = Counter()
    for a in A:
        for b in A:
            counts[(a + b) % n] += 1
    total = len(A) ** 2
    return {s: c / total for s, c in counts.items()}


n = 8
for A in ([0, 1, 2], [0, 2, 4, 6], [0, 1, 2, 3]):
    A_plus_A = sorted({(a + b) % n for a in A for b in A})
    somme = distribution_somme(A, n)
    h_somme = entropy_bits(somme)
    h_x = math.log2(len(A))
    print(f"A = {A}  |A| = {len(A)}  |A+A| = {len(A_plus_A)} {A_plus_A}")
    print(f"   H(X) = {h_x:.4f}   H(X+X') = {h_somme:.4f}   log2|A+A| = {math.log2(len(A_plus_A)):.4f}"
          f"   inegalite = {h_somme <= math.log2(len(A_plus_A)) + 1e-12}")
    print(f"   H(X+X') - H(X) = {h_somme - h_x:+.4f}  <= log2 K = "
          f"{math.log2(len(A_plus_A) / len(A)):+.4f}")
    print()

A = [0, 1, 2]  |A| = 3  |A+A| = 5 [0, 1, 2, 3, 4]
   H(X) = 1.5850   H(X+X') = 2.1972   log2|A+A| = 2.3219   inegalite = True
   H(X+X') - H(X) = +0.6122  <= log2 K = +0.7370

A = [0, 2, 4, 6]  |A| = 4  |A+A| = 4 [0, 2, 4, 6]
   H(X) = 2.0000   H(X+X') = 2.0000   log2|A+A| = 2.0000   inegalite = True
   H(X+X') - H(X) = +0.0000  <= log2 K = +0.0000

A = [0, 1, 2, 3]  |A| = 4  |A+A| = 7 [0, 1, 2, 3, 4, 5, 6]
   H(X) = 2.0000   H(X+X') = 2.6556   log2|A+A| = 2.8074   inegalite = True
   H(X+X') - H(X) = +0.6556  <= log2 K = +0.8074



### Lecture du résultat

Les trois ensembles racontent trois choses différentes, et c'est leur comparaison qui fait la marche :

- **`A = {0, 1, 2}`** — `|A + A| = 5`, donc `log₂ 5 = 2,3219`, tandis que `H(X + X') = 2,1972`. L'inégalité est **stricte** : la somme n'est pas uniforme sur `A + A` (les bords sont moins probables que le centre), donc elle n'atteint pas la borne. C'est le cas qui compte pédagogiquement — `H(X + X') ≤ log |A + A|` est une borne, pas une identité, et le déficit mesure exactement la non-uniformité de la somme.
- **`A = {0, 2, 4, 6}`** — les éléments pairs de `Z/8Z`, c'est-à-dire un vrai sous-groupe, isomorphe à `Z/4Z`. La somme de deux uniformes sur un sous-groupe est uniforme sur ce sous-groupe, donc `A + A = A` et `H(X + X') = H(X) = 2` bits : l'inégalité est **atteinte**, et `log₂ K = 0`. C'est le régime de structure parfaite — aucune croissance du support, aucune incertitude ajoutée par la somme.
- **`A = {0, 1, 2, 3}`** — `|A + A| = 7` et `H(X + X') = 2,6556`, pour `H(X) = 2`. La croissance existe (`log₂ 7 = 2,8074`) mais la borne n'est pas saturée, et `H(X + X') − H(X) = 0,6556` reste sous `log₂ K = 0,8074` : la forme entropique de l'hypothèse de croissance tient.

Un piège à signaler, parce qu'il est facile à commettre en lisant la table : `{0, 2, 4}` n'est **pas** un sous-groupe de `Z/8Z` (le sous-groupe pair est `{0, 2, 4, 6}`). Sa somme vaut `{0, 2, 4, 6}`, donc son support **double** et l'inégalité y est stricte. Un ensemble dont la somme « ressemble » à lui-même n'est pas pour autant un sous-groupe — c'est exactement le genre d'écart que la borne entropique mesure et que l'œil rate.

**Ce que le calcul n'établit pas, et qui reste au lac** : rien ici ne prouve l'inégalité de Ruzsa, ni le théorème de réduction de distance, ni la conjecture PFR. Le calcul vérifie que **le pont se comporte comme annoncé** sur trois cas, ce qui est la seule chose qu'un calcul fini puisse faire. La chaîne complète — de `H(X + X') ≤ log |A + A|` à « `A` est contenu dans un nombre polynomial de translatés de sous-groupes » — est l'objet d'ANALYSE-03, et sa version formelle celui du lac `teorth/pfr`.

**Où reprendre** : la section 3 d'ANALYSE-03 écrit la conjecture entropique et la distance de Ruzsa ; la section 1 d'ANALYSE-04 en fait une primitive avec ses tests de limite. Les deux supposent désormais acquise la matière de ce carnet.


## 6. Exercices

### Exercice 1 — L'entropie d'une loi qui n'est pas uniforme

La section 2 a vérifié `H = log |A|` pour une loi **uniforme**. Le cas non uniforme est celui qui distingue les deux notions, et il est utile de le mesurer.

Prenez la loi `p = {0: 0.5, 1: 0.25, 2: 0.125, 3: 0.125}` sur `A = {0, 1, 2, 3}`.

- **Étape 1** : calculez `H(p)` avec `entropy_bits`.
- **Étape 2** : calculez `log₂ |A|` et l'écart `log₂ |A| − H(p)`.
- **Étape 3** : vérifiez que l'écart est bien égal à la divergence `D(p ‖ uniforme) = Σ p · log₂(p / (1/|A|))`, et concluez sur ce que mesure cet écart.

**Indice** : `H ≤ log |A|` avec égalité si et seulement si la loi est uniforme — l'écart est donc une mesure d'éloignement à l'uniforme, et c'est précisément la divergence de Kullback-Leibler vers la loi uniforme.


In [7]:
# Exercice 1 -- loi non uniforme, ecart a log|A|, divergence vers l'uniforme (a completer)
p_non_uniforme = {0: 0.5, 1: 0.25, 2: 0.125, 3: 0.125}


def ecart_a_uniforme(p):
    """Retourne (H(p), log2|support|, ecart) -- Etapes 1 et 2."""
    pass  # TODO etudiant (Etapes 1-2)


def divergence_vers_uniforme(p):
    """D(p || uniforme) = somme p log2(p / (1/|support|)) -- Etape 3."""
    pass  # TODO etudiant (Etape 3)


print("Exercice a completer : H(p), log2|A|, ecart, puis divergence vers l'uniforme.")
print("Attendu : ecart == divergence, et H(p) strictement inferieur a log2(4) = 2 bits.")

Exercice a completer : H(p), log2|A|, ecart, puis divergence vers l'uniforme.
Attendu : ecart == divergence, et H(p) strictement inferieur a log2(4) = 2 bits.


### Exercice 2 — La règle de chaîne sur trois variables

La section 3 a vérifié `H(X, Y) = H(X) + H(Y|X)` sur deux variables. La forme à trois variables est celle que les preuves utilisent réellement, sous la forme `H(X, Y, Z) = H(X) + H(Y|X) + H(Z | X, Y)`.

Reprenez le canal binaire de la section 3 avec `q = 0.9`, et posez `Z = X XOR Y` (l'indicatrice du bruit du canal).

- **Étape 1** : construisez la loi jointe `(X, Y, Z)`.
- **Étape 2** : calculez `H(X)`, `H(Y|X) = H(X,Y) − H(X)` et `H(Z | X, Y) = H(X,Y,Z) − H(X,Y)`.
- **Étape 3** : vérifiez que la somme des trois vaut `H(X, Y, Z)`, puis expliquez pourquoi `H(Z | X, Y)` doit valoir exactement `0`.

**Indice** : `Z` est une fonction déterministe de `(X, Y)`, donc la connaître tous les deux détermine `Z` sans reste. Le résultat attendu n'est pas une petite valeur, c'est un zéro exact — et c'est une illustration directe de « `H(W|V) = 0` ⟺ `V` détermine `W` ».


In [8]:
# Exercice 2 -- regle de chaine a trois variables, Z = X xor Y (a completer)
q = 0.9


def joint_xyz(q):
    """Loi jointe (x, y, x xor y) pour le canal binaire symetrique de fidelite q."""
    pass  # TODO etudiant (Etape 1)


def chaine_trois_variables(joint3):
    """Retourne (H(X), H(Y|X), H(Z|X,Y), H(X,Y,Z)) -- Etape 2."""
    pass  # TODO etudiant (Etape 2)


print("Exercice a completer : decomposer H(X,Y,Z) en trois termes conditionnels.")
print("Attendu : H(Z|X,Y) = 0 exactement, et la somme des trois termes = H(X,Y,Z).")

Exercice a completer : decomposer H(X,Y,Z) en trois termes conditionnels.
Attendu : H(Z|X,Y) = 0 exactement, et la somme des trois termes = H(X,Y,Z).


### Exercice 3 — Trouver un ensemble où la borne de la section 5 est atteinte

La section 5 a mesuré trois ensembles et trouvé un cas **strict** (`A = {0, 1, 2}`) et un cas d'**égalité** (`A = {0, 2, 4, 6}`, le sous-groupe pair). La question ouverte est de savoir ce qui décide.

Cherchez, par énumération sur `Z/8Z`, tous les ensembles `A` de cardinal 3, et pour chacun :

- **Étape 1** : calculez `H(X + X')` et `log₂ |A + A|`.
- **Étape 2** : classez les ensembles selon que l'inégalité est stricte ou atteinte.
- **Étape 3** : regardez ce que les ensembles du second groupe ont en commun, et formulez une conjecture — puis cherchez un contre-exemple avant de la retenir.

**Indice** : l'égalité `H(X + X') = log |A + A|` exige que la somme soit **uniforme** sur `A + A`. C'est automatique quand `A` est un sous-groupe de `Z/nZ` (ou un translaté d'un sous-groupe), exceptionnel sinon. Or `Z/8Z` n'a **aucun** sous-groupe d'ordre 3 — `8 = 2³` — donc la question reste ouverte pour les 56 ensembles : la réponse n'est pas dans la théorie, elle est dans l'énumération. Commencez par comparer `{0, 1, 2}` (section 5) à `{1, 2, 3}`, son translaté : si l'égalité ne dépendait que de la forme, les deux se comporteraient pareil. **Contrôle** : refaites l'énumération dans `Z/9Z`, où `{0, 3, 6}` est cette fois un vrai sous-groupe d'ordre 3 — l'égalité doit y être atteinte, et c'est le témoin positif qui valide votre critère.


In [9]:
# Exercice 3 -- enumeration des ensembles de cardinal 3 dans Z/8Z (a completer)
from itertools import combinations

n = 8


def classer_ensembles(n, k):
    """Pour chaque A de cardinal k, retourne (A, H(X+X'), log2|A+A|, ecart) -- Etapes 1-2."""
    pass  # TODO etudiant (Etapes 1-2)


print("Exercice a completer : classer les C(8,3) = 56 ensembles de Z/8Z selon l'ecart.")
print("Attendu : deux groupes possibles ; les cas d'egalite exigent une somme uniforme sur A+A.")

Exercice a completer : classer les C(8,3) = 56 ensembles de Z/8Z selon l'ecart.
Attendu : deux groupes possibles ; les cas d'egalite exigent une somme uniforme sur A+A.


## 7. Conclusion

### 7.1 Ce que la marche achève

Ce carnet a posé, calculé et vérifié les quatre objets dont ANALYSE-03 a besoin sans les définir :

| Objet | Où il a été établi | Ce qui a été mesuré, pas seulement énoncé |
|---|---|---|
| `H(X) = −Σ p log p`, bornes, unités | §1 | table de `H₂`, rapport bits/nats valant `ln 2` |
| `H(uniforme sur A) = log \|A\|` | §2 | trois groupes d'ordre 8 à la même valeur, un ensemble sans structure à `log 3` |
| règle de chaîne `H(X,Y) = H(X) + H(Y\|X)` | §3 | vérifiée au bit près, et `H(Y\|X) = H₂(q)` identifiée |
| sous-modularité, `I(X;Y)`, cas d'égalité | §4 | trois régimes, dont le critère d'indépendance et le plafond `I ≤ min(H(X), H(Y))` |
| pont `H(X + X') ≤ log \|A + A\|` | §5 | borne stricte sur `{0,1,2}`, atteinte sur le sous-groupe `{0,2,4,6}`, et `{0,2,4}` qui n'en est pas un |

### 7.2 Ce que la marche ne fait pas

Elle ne remplace aucune preuve. Chaque vérification porte sur des lois finies choisies à la main, ce qui donne des certitudes sur des cas — jamais un théorème. En particulier, rien ici n'établit l'inégalité de Ruzsa, le théorème de réduction de distance, ni PFR : ce sont des énoncés universels, et un calcul fini ne les atteint pas. La section 5 l'a dit à chaque étape, et c'est la raison d'être de la sous-série : le calcul prépare la lecture, le lac porte la garantie.

### 7.3 Où continuer

| Suite | Ce qu'on y trouve |
|---|---|
| [ANALYSE-03 — PFR](ANALYSE-03-PFR-Lean.ipynb) | la conjecture en version combinatoire **et** entropique, la distance de Ruzsa, la méthode entropique et le lac `teorth/pfr` |
| [ANALYSE-04 — Primitives](ANALYSE-04-PFR-Primitives-Python.ipynb) | la distance de Ruzsa comme primitive transportable, ses tests de limite et le contre-exemple `Z/4Z` sur `d[X;X]` |
| [Lean-20 — Capstone](../Lean-20-Capstone-Digestions-Tao-Python.ipynb) | l'escalier depuis la série principale : règle de chaîne et distance de Ruzsa sur `F₂³` |
